In [15]:
# get table
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
utils.fetch_data_from_postgres_via_psycopg2(
  """
  SELECT table_name
  FROM information_schema.tables
  WHERE table_schema = 'public';
"""
)
# user_id: [1, 6040]
# movie_id: [1, 3706]

,table_name
0,tpcxai_order_training
1,tpcxai_product_serving
2,tpcxai_financial_transactions_serving
3,tpcxai_store_dept_serving
4,tpcxai_order_serving
5,tpcxai_financial_account_training
6,tpcxai_lineitem_training
7,tpcxai_financial_account_serving
8,tpcxai_product_rating_training
9,tpcxai_lineitem_serving


In [ ]:
# template 6
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder


# Use case 8, trainig query
df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday,     -- Equivalent to np.min(x) for weekday
        MIN(trip_type) AS trip_type                 -- Equivalent to np.min(x) for trip_type
    FROM tpcxai_order_training 
    JOIN tpcxai_lineitem_training ON o_order_id = li_order_id 
    JOIN tpcxai_product_training ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
""")

le_department = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_training_data['department_encoded'] = le_department.fit_transform(df_training_data[['department']])
le_trip_type = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_training_data['trip_type_encoded'] = le_trip_type.fit_transform(df_training_data[['trip_type']])

X_features = df_training_data[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y = df_training_data['trip_type_encoded'].values.astype(float)
num_y = len(np.unique(y))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_features.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_y, activation='softmax')  # Output layer for trip_type
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X_train, X_val, y_train, y_val = train_test_split(X_features, y, test_size=0.2, random_state=42)
# model.fit(X_train, y_train, epochs=10, batch_size=2048, validation_data=(X_val, y_val))

In [ ]:
# testing part
df_test_data = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday     -- Equivalent to np.min(x) for weekday
    FROM tpcxai_order_serving 
    JOIN tpcxai_lineitem_serving ON o_order_id = li_order_id 
    JOIN tpcxai_product_serving ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
""")

df_test_data['department_encoded'] = le_department.transform(df_test_data[['department']])
X_infer = df_test_data[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y_pred = model.predict(X_infer, batch_size=2048)


In [ ]:
# encoder 
print("Ordinal Encoder for Department:", le_department.categories_)
print("Ordinal Encoder for Trip Type:", le_trip_type.categories_)

Ordinal Encoder for Department: [array(['AUTOMOTIVE', 'BATH AND SHOWER', 'BEAUTY', 'BEDDING', 'BOYS WEAR',
       'CANDY, TOBACCO, COOKIES', 'CELEBRATION', 'COMM BREAD',
       'COOK AND DINE', 'DAIRY', 'DSD GROCERY', 'ELECTRONICS',
       'FABRICS AND CRAFTS', 'FINANCIAL SERVICES', 'FROZEN FOODS',
       'GIRLS WEAR, 4-6X  AND 7-14', 'GROCERY DRY GOODS', 'HARDWARE',
       'HOME DECOR', 'HOME MANAGEMENT', 'HORTICULTURE AND ACCESS',
       'HOUSEHOLD CHEMICALS/SUPP', 'HOUSEHOLD PAPER GOODS',
       'IMPULSE MERCHANDISE', 'INFANT APPAREL',
       'INFANT CONSUMABLE HARDLINES', 'JEWELRY AND SUNGLASSES',
       'LADIESWEAR', 'LAWN AND GARDEN', 'LIQUOR,WINE,BEER',
       'MEAT - FRESH & FROZEN', 'MEDIA AND GAMING', 'MENS WEAR',
       'OFFICE SUPPLIES', 'PAINT AND ACCESSORIES', 'PERSONAL CARE',
       'PETS AND SUPPLIES', 'PHARMACY OTC', 'PHARMACY RX',
       'PLAYERS AND ELECTRONICS', 'PRODUCE', 'SERVICE DELI', 'SHOES',
       'SPORTING GOODS', 'TOYS', 'WIRELESS'], dtype=object)]
Ordinal 

In [1]:
# template 7
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder

# load data
# df_user = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")
# df_movie = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_movie""")
# df_rating = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_rating""")

# trainig query
df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm, is_fraud
from tpcxai_financial_account_training join tpcxai_financial_transactions_training on fa_customer_sk=sender_id
""")

2025-06-19 05:34:23.343785: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 05:34:23.386494: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-19 05:34:23.386531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-19 05:34:23.387738: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 05:34:23.395036: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
X_features = df_training_data[['business_hour_norm', 'amount_norm']].values.astype(float)
y = df_training_data['is_fraud'].values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)
# train DNN for fraud detection
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(X_train, y_train, epochs=10, batch_size=512, validation_data=(X_test, y_test))

In [17]:
# testing part
df_test_data = utils.fetch_data_from_postgres_via_psycopg2(
    """
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm
from tpcxai_financial_account_serving 
join tpcxai_financial_transactions_serving on fa_customer_sk=sender_id
"""
)
X_infer = df_test_data[["business_hour_norm", "amount_norm"]].values.astype(float)
y_pred = model.predict(X_infer, batch_size=512)

1600/1600 [==============================] - 1s 814us/step


In [1]:
# template 8
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder

df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
select store, department, li_order_id, price, quantity,
EXTRACT(WEEK FROM date) AS week,
EXTRACT(MONTH FROM date) AS month,
CASE 
    WHEN EXTRACT(WEEK FROM date) > 50 AND EXTRACT(MONTH FROM date) = 1 
    THEN EXTRACT(YEAR FROM date) - 1
    ELSE EXTRACT(YEAR FROM date)
END AS year,
quantity * price as row_price
from tpcxai_order_training join tpcxai_lineitem_training on o_order_id=li_order_id
join tpcxai_product_training on li_product_id=p_product_id
""")

2025-06-19 05:52:41.567345: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 05:52:41.609768: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-19 05:52:41.609805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-19 05:52:41.611016: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 05:52:41.618286: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
grouped = df_training_data.groupby(['store', 'department', 'year', 'week'])['row_price'].sum().reset_index()
grouped = grouped.rename(index=str, columns={'store': 'Store', 'department': 'Dept', 'date': 'Date', 'row_price': 'Weekly_Sales'})
grouped['num_of_week'] = (grouped['year'].astype(int) - 2010) * 52 + grouped['week'].astype(int) - 1
le_store = LabelEncoder()
le_dept = LabelEncoder()

grouped['Store'] = le_store.fit_transform(grouped['Store'])
grouped['Dept'] = le_dept.fit_transform(grouped['Dept'])
min_num_of_week = 0
max_num_of_week = 52*3
grouped['num_of_week'] = (grouped['num_of_week'] - 0) / max_num_of_week
X_features = grouped[['Store', 'Dept', 'num_of_week']].values
y = grouped['Weekly_Sales'].values
y_min = y.min()
y_max = y.max()
y = (y - y_min) / (y_max - y_min)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(3,)),
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)
model.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_test, y_test))

In [ ]:
# testing part
df_testing_data = utils.fetch_data_from_postgres_via_psycopg2("""
select store, department, num_of_week from tpcxai_store_dept_serving
""")
df_testing_data['store'] = le_store.transform(df_testing_data['store'].values)
df_testing_data['department'] = le_dept.transform(df_testing_data['department'].values)
df_testing_data['num_of_week'] = (df_testing_data['num_of_week'] - 0) / max_num_of_week
X_serve = df_testing_data[['store', 'department', 'num_of_week']].values.astype(float)
y_pred = model.predict(X_serve)
y_pred = y_pred * (y_max - y_min) + y_min

In [7]:
print("Label Encoder for Store:", le_store.classes_)
print("Label Encoder for Department:", le_dept.classes_)

Label Encoder for Store: [ 1  2  3  4  5  6  7  8  9 10 11]
Label Encoder for Department: ['AUTOMOTIVE' 'BATH AND SHOWER' 'BEAUTY' 'BEDDING' 'BOYS WEAR'
 'CANDY, TOBACCO, COOKIES' 'CELEBRATION' 'COMM BREAD' 'COOK AND DINE'
 'DAIRY' 'DSD GROCERY' 'ELECTRONICS' 'FABRICS AND CRAFTS'
 'FINANCIAL SERVICES' 'FROZEN FOODS' 'GIRLS WEAR, 4-6X  AND 7-14'
 'GROCERY DRY GOODS' 'HARDWARE' 'HOME DECOR' 'HOME MANAGEMENT'
 'HORTICULTURE AND ACCESS' 'HOUSEHOLD CHEMICALS/SUPP'
 'HOUSEHOLD PAPER GOODS' 'IMPULSE MERCHANDISE' 'INFANT APPAREL'
 'INFANT CONSUMABLE HARDLINES' 'JEWELRY AND SUNGLASSES' 'LADIESWEAR'
 'LAWN AND GARDEN' 'LIQUOR,WINE,BEER' 'MEAT - FRESH & FROZEN'
 'MEDIA AND GAMING' 'MENS WEAR' 'OFFICE SUPPLIES' 'PAINT AND ACCESSORIES'
 'PERSONAL CARE' 'PETS AND SUPPLIES' 'PHARMACY OTC' 'PHARMACY RX'
 'PLAYERS AND ELECTRONICS' 'PRODUCE' 'SERVICE DELI' 'SHOES'
 'SPORTING GOODS' 'TOYS' 'WIRELESS']
